# Your First Hybrid Job

Build a Bell state locally, then package it as a managed Amazon Braket Hybrid Job and walk the create -> queue -> run -> complete lifecycle.

**Objectives:**
- Build and run a parameterized Bell-state circuit on the `LocalSimulator` and read its |00>/|11> correlation
- Write a job entry-point script that builds the circuit and calls `save_job_result`
- Read the `AwsQuantumJob.create(...)` / `job.state()` / `job.result()` lifecycle as code, gated behind `RUN_ON_AWS`
- Explain when a Hybrid Job earns its keep over a standalone task

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

In [ ]:
# Setup: run this cell first.
# Requires: pip install -e '.[dev]' from the project root (see `make setup`).
#
# Importing braket.aws / braket.jobs is free and needs NO AWS credentials.
# Nothing in this notebook calls AWS at import time. Every AWS call below is
# guarded behind `RUN_ON_AWS = False` with a cost note.
import numpy as np
import matplotlib.pyplot as plt

from braket.circuits import Circuit, FreeParameter
from braket.devices import LocalSimulator

# These imports succeed without credentials; we only *call* them under RUN_ON_AWS.
from braket.aws import AwsQuantumJob
from braket.jobs import save_job_result

np.random.seed(7)

# Free, instant, no credentials. All real output in this notebook comes from here.
device = LocalSimulator()
print("LocalSimulator ready:", device)

## 1. Build and run the Bell state locally

Before packaging anything for AWS, prove the circuit on your laptop. The Bell state
$|\Phi^+\rangle = \tfrac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ is the canonical two-qubit entangled state: a Hadamard on qubit 0 puts it in superposition, then a CNOT copies that into qubit 1. Measuring it yields only `00` or `11` -- never `01` or `10` -- because the two qubits are perfectly correlated.

In [ ]:
bell = Circuit().h(0).cnot(0, 1)
print(bell)

result = device.run(bell, shots=1000).result()
counts = dict(result.measurement_counts)
print("\ncounts:", counts)

# The defining property: only the correlated outcomes appear.
assert set(counts).issubset({"00", "11"}), counts
print("Only |00> and |11> observed -- the qubits are entangled.")

In [ ]:
# Visualize the correlation.
fig, ax = plt.subplots(figsize=(4.5, 3))
labels = ["00", "01", "10", "11"]
values = [counts.get(b, 0) for b in labels]
ax.bar(labels, values, color=["#33bb66", "#cccccc", "#cccccc", "#33bb66"])
ax.set_xlabel("measured bitstring")
ax.set_ylabel("counts (1000 shots)")
ax.set_title("Bell state: only |00> and |11>")
plt.tight_layout()
plt.show()

## 2. Parameterize it -- the inner loop of every hybrid job

A real hybrid job does not submit a fixed circuit once; it submits the *same circuit
structure* every iteration with a fresh angle chosen by a classical optimizer. Declare
the angle as a `FreeParameter` and pass values through `inputs={...}` at run time. This
is the seed of *parametric compilation*: one structure, many parameter values.

Here `Rx(theta)` on qubit 0 followed by a CNOT sweeps continuously from the product
state `00` (theta=0), through a balanced entangled state (theta=pi/2), to the flipped
product state `11` (theta=pi).

In [ ]:
theta = FreeParameter("theta")
param_bell = Circuit().rx(0, theta).cnot(0, 1)
print(param_bell)

for t in [0.0, np.pi / 2, np.pi]:
    r = device.run(param_bell, shots=2000, inputs={"theta": float(t)}).result()
    print(f"theta={t:5.3f}  ->  {dict(r.measurement_counts)}")

## 3. A local variational loop -- what the job will actually run

The classical half of a hybrid algorithm reads a measurement, computes a cost, and picks
the next angle. We run that loop here locally so you see real convergence with no AWS.

The cost is $\langle Z_0 \rangle$ -- the expectation of $Z$ on qubit 0, estimated from
shot counts as $(n_0 - n_1)/N$. Minimizing it drives qubit 0 toward $|1\rangle$, i.e.
$\theta \to \pi$. We use a finite-difference gradient and plain gradient descent. This
is intentionally tiny (1 qubit, 12 steps) so it runs in a second -- but it is structurally
the same loop a VQE or QAOA job runs for hundreds of iterations.

In [ ]:
one_qubit = Circuit().rx(0, FreeParameter("theta"))

def expval_z0(angle, shots=4000):
    """Estimate <Z0> from shot counts: (#|0> - #|1>) / N."""
    c = dict(device.run(one_qubit, shots=shots, inputs={"theta": float(angle)}).result().measurement_counts)
    n0, n1 = c.get("0", 0), c.get("1", 0)
    return (n0 - n1) / (n0 + n1)

angle = 0.3          # initial guess
lr = 0.6             # learning rate
eps = 0.1            # finite-difference step
cost_history = []

for step in range(12):
    grad = (expval_z0(angle + eps) - expval_z0(angle - eps)) / (2 * eps)
    cost = expval_z0(angle)
    cost_history.append(cost)
    angle = angle - lr * grad

print("final theta:", round(angle, 3), " target pi =", round(float(np.pi), 3))
print("cost went from", round(cost_history[0], 3), "to", round(cost_history[-1], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(len(cost_history)), cost_history, "o-", color="#3366cc")
ax.axhline(-1.0, ls="--", color="#999999", label="ground state <Z0> = -1")
ax.set_xlabel("iteration")
ax.set_ylabel("cost  <Z0>")
ax.set_title("Local variational loop converges")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Package the algorithm: the entry-point script

A Hybrid Job runs an **entry-point script** inside a managed container on AWS. The script
is ordinary Python: it builds the circuit, runs it (on the job's device), and calls
`save_job_result(...)` to persist outputs to S3. Inside a real job the device comes from
the `AMZN_BRAKET_DEVICE_ARN` environment variable that Braket injects; locally we fall
back to `LocalSimulator` so the *same script* is runnable on your laptop.

We define the entry point as a **string** and write it to a file. Defining and compiling a
string touches no AWS -- we even run it locally below to prove it works before any job exists.

In [ ]:
ENTRY_POINT = """
import os

from braket.circuits import Circuit
from braket.devices import LocalSimulator
from braket.jobs import save_job_result


def get_device():
    # Inside a Hybrid Job, Braket sets AMZN_BRAKET_DEVICE_ARN. Run the same
    # script locally by falling back to the LocalSimulator when it is absent.
    arn = os.environ.get("AMZN_BRAKET_DEVICE_ARN")
    if arn and arn.startswith("arn:"):
        from braket.aws import AwsDevice
        return AwsDevice(arn)
    return LocalSimulator()


def main():
    shots = int(os.environ.get("SHOTS", "1000"))
    device = get_device()

    bell = Circuit().h(0).cnot(0, 1)
    counts = dict(device.run(bell, shots=shots).result().measurement_counts)

    p_correlated = (counts.get("00", 0) + counts.get("11", 0)) / shots
    save_job_result({"counts": counts, "p_correlated": p_correlated})
    print("counts:", counts, "p_correlated:", p_correlated)


if __name__ == "__main__":
    main()
"""

with open("first_job_entry.py", "w") as f:
    f.write(ENTRY_POINT)

print("Wrote first_job_entry.py (", len(ENTRY_POINT), "chars )")

In [ ]:
# Sanity-check the entry point WITHOUT AWS: run it as a subprocess against the
# LocalSimulator (AMZN_BRAKET_DEVICE_ARN unset -> falls back to LocalSimulator).
# save_job_result is a no-op outside a real job, so this exercises the real code path
# end-to-end with no credentials.
import os
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "first_job_entry.py"],
    capture_output=True, text=True,
    env={**os.environ, "SHOTS": "500"},
)
print("exit code:", proc.returncode)
print(proc.stdout.strip())
if proc.returncode != 0:
    print("STDERR:", proc.stderr[-500:])

## 5. Submit it as a Hybrid Job (gated -- needs AWS)

This is the only part that touches AWS. `AwsQuantumJob.create(...)` packages the entry
point, uploads it, provisions a classical instance, and runs your script with **priority
access** to the device. You then poll `job.state()` through the lifecycle and pull
`job.result()` from S3 when it reaches `COMPLETED`.

The lifecycle is fixed: **create -> QUEUED -> RUNNING -> COMPLETED** (or `FAILED` /
`CANCELLED`). Everything below is guarded behind `RUN_ON_AWS = False` so this cell is inert
by default.

In [ ]:
RUN_ON_AWS = False  # Flip to True ONLY with AWS credentials configured.

# COST NOTE: A Hybrid Job bills the classical instance per hour (ml.m5.large ~ $0.10-$0.30/hr)
# PLUS the quantum tasks. On the local device below the quantum side is free and the run is
# seconds, so cost is dominated by the brief instance time. Switching `device` to a real QPU
# ARN adds per-shot + per-task charges. Always prototype locally first (see cells 1-4).

if RUN_ON_AWS:
    import time

    job = AwsQuantumJob.create(
        device="local:braket/braket.local.qubit",  # free local device inside the managed instance
        source_module="first_job_entry.py",
        entry_point="first_job_entry:main",
        job_name="first-hybrid-bell",
        hyperparameters={},  # the script reads SHOTS from the environment
        wait_until_complete=False,
    )
    print("created:", job.arn)

    # Poll the lifecycle: QUEUED -> RUNNING -> COMPLETED.
    terminal = {"COMPLETED", "FAILED", "CANCELLED"}
    state = job.state()
    while state not in terminal:
        print("state:", state)
        time.sleep(15)
        state = job.state()
    print("final state:", state)

    if state == "COMPLETED":
        result = job.result()  # downloads from S3
        print("result:", result)
else:
    print("RUN_ON_AWS is False -- no AWS calls made.")
    print("Local results above already prove the circuit and the entry point work.")

## 6. When does a job beat a standalone task?

A standalone `device.run(circuit)` task is fire-and-forget: it joins the back of the
device's general queue. For one circuit that is fine. The break-even is **iteration**. A
variational loop where each step depends on the last pays the queue wait *once per step* as
a standalone task, with the classical optimizer idle in between. A Hybrid Job pays the
queue wait essentially once: its tasks carry a job token granting priority, so iterations
run back-to-back.

Rough rule of thumb: `standalone_wall ~ iterations * (queue_wait + run)` versus
`job_wall ~ startup + iterations * run`. The job's fixed startup is amortized once
iterations and queue wait are large.

In [ ]:
def standalone_wall(iterations, queue_wait, run_sec):
    return iterations * (queue_wait + run_sec)

def job_wall(iterations, run_sec, startup=180.0):
    # priority access -> queue paid ~once, folded into the fixed startup
    return startup + iterations * run_sec

iters = np.arange(1, 201)
queue_wait, run_sec = 45.0, 6.0
standalone = np.array([standalone_wall(n, queue_wait, run_sec) / 60 for n in iters])
job = np.array([job_wall(n, run_sec) / 60 for n in iters])

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(iters, standalone, label="standalone tasks", color="#cc3333")
ax.plot(iters, job, label="hybrid job", color="#3366cc")
ax.set_xlabel("iterations")
ax.set_ylabel("wall-clock (minutes)")
ax.set_title("Job wins once iteration count is high")
ax.legend()
plt.tight_layout()
plt.show()

# Break-even iteration count.
crossover = int(iters[np.argmax(standalone > job)])
print("standalone overtakes the job at ~", crossover, "iterations (queue_wait=45s, run=6s)")

## Exercises

Four exercises to cement the loop. Each has two tiered hints -- expand only what you need -- and a check cell that tells you when you have it. Work them on the `LocalSimulator` first: Exercises 1-3 and the timing helper in Exercise 4 all run with no credentials, and you only flip `RUN_ON_AWS = True` (with credentials and a cost check) to submit the job itself.

### Exercise 1 — Sweep the entangling angle

The parameterized circuit `param_bell` (`Rx(theta)` then `CNOT`) has one knob. Sweep it across the full range and watch the correlated outcomes.

Define `theta_grid`: 9 angles evenly spaced from 0 to pi (inclusive). Define `p_corr`: the measured probability of a *correlated* outcome, p(00) + p(11), at each angle, in the same order, from at least 1000 shots per point. Plot it if you like -- but predict the shape first: does p(correlated) actually move as theta changes, and what does that say about what the `CNOT` is doing?

<details><summary>Hint 1 — nudge</summary>

Section 2 already ran `param_bell` at three angles; you are doing the same at nine and summing the counts that land on the two *agreeing* bitstrings. The `CNOT` ties qubit 1 to qubit 0 no matter what `Rx` did first.

</details>
<details><summary>Hint 2 — approach</summary>

Build the angles with `np.linspace(0, np.pi, 9)`. Loop over them; for each, call `device.run(param_bell, shots=..., inputs={"theta": float(t)})`, read `.result().measurement_counts`, and append `(counts["00"] + counts["11"]) / total` to `p_corr`. Use `counts.get(b, 0)` so a missing bitstring counts as zero.

</details>

In [ ]:
# Exercise 1: Sweep theta from 0 to pi and measure p(correlated) = p(00) + p(11).
# Define: theta_grid -- 9 angles from 0 to pi; p_corr -- the correlated
#         probability at each angle, same order (>= 1000 shots per point).

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(theta_grid) == 9, "sweep theta in 9 steps"
    assert abs(theta_grid[0]) < 1e-9, "the sweep should start at theta = 0"
    assert abs(theta_grid[-1] - np.pi) < 1e-6, "the sweep should end at theta = pi"
    assert len(p_corr) == 9, "record one correlated-probability per angle"
    assert all(0.0 <= p <= 1.0 for p in p_corr), "probabilities live in [0, 1]"
    assert all(p > 0.98 for p in p_corr), (
        "Rx(theta)+CNOT only ever populates |00> and |11>, so p(correlated) "
        "stays ~1 across the whole sweep -- the CNOT locks the correlation"
    )

### Exercise 2 — A two-qubit cost, and the limits of an ansatz

The local loop in Section 3 minimized `<Z0>`. Swap the cost for the Bell correlation `<Z0 Z1>` and re-run the descent on the two-qubit `Rx(theta)+CNOT` circuit. Before you run it, predict: can this ansatz make the two qubits *anti*-correlated, and so what will the optimizer be able to do?

Define `expval_zz`: a function taking an angle and returning the estimated `<Z0 Z1>` from that circuit, computed from counts as `(n00 + n11 - n01 - n10) / N`. Define `zz_history`: the cost recorded at each iteration of a short gradient-descent loop over the angle.

<details><summary>Hint 1 — nudge</summary>

`<Z0 Z1>` is +1 whenever the two qubits agree and -1 whenever they disagree. Look at which bitstrings `Rx(theta)+CNOT` can ever produce. If the disagreeing outcomes have zero probability for *every* theta, what is `<Z0 Z1>`, and what can gradient descent do about it?

</details>
<details><summary>Hint 2 — approach</summary>

Mirror `expval_z0` from Section 3, but on `Circuit().rx(0, theta).cnot(0, 1)`, mapping the four two-qubit bitstrings to +/-1 as above. Then reuse the same finite-difference gradient and update rule from that loop, appending the cost to `zz_history` each step.

</details>

In [ ]:
# Exercise 2: Re-run the local descent with the Bell-correlation cost <Z0 Z1>.
# Define: expval_zz(angle) -- estimated <Z0 Z1> from Rx(theta)+CNOT as
#         (n00 + n11 - n01 - n10) / N; zz_history -- the cost at each descent step.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert callable(expval_zz), "expval_zz should be a function of the angle"
    for _a in (0.0, np.pi / 2, np.pi):
        _v = expval_zz(_a)
        assert -1.0 <= _v <= 1.0, "a Z0 Z1 expectation lives in [-1, 1]"
        assert _v > 0.9, (
            "every shot of Rx(theta)+CNOT lands on |00> or |11>, both with "
            "Z0 Z1 = +1, so the correlation reads ~+1 at every angle"
        )
    assert len(zz_history) >= 5, "record the cost at each descent iteration"
    assert all(-1.0 <= v <= 1.0 for v in zz_history), (
        "each recorded cost is a Z0 Z1 expectation in [-1, 1]"
    )
    assert all(v > 0.9 for v in zz_history), (
        "this ansatz cannot make the qubits disagree, so the cost never falls "
        "below +1 however the angle moves"
    )

### Exercise 3 — A hyperparameter in the entry point

A real job is tuned by hyperparameters the container reads from its environment. Extend the entry-point script so it reads `N_LAYERS` from the environment, stacks that many `H`+`CNOT` layers, and records the layer count in its saved result alongside the counts. Then re-run the local subprocess check (no AWS) to prove the edited script still runs end-to-end.

Define `layered_result`: the parsed result of running your edited entry point as a subprocess with a layer count of at least 2 -- a dict carrying the `counts` and an `n_layers` field.

<details><summary>Hint 1 — nudge</summary>

Section 4 already reads `SHOTS` from `os.environ` inside `main()`. A second hyperparameter is the same move: read it, default it, and let it drive how many times you append `.h(0).cnot(0, 1)` to the circuit.

</details>
<details><summary>Hint 2 — approach</summary>

Edit the `ENTRY_POINT` string: add `n_layers = int(os.environ.get("N_LAYERS", "1"))`, build the circuit in a loop, and pass `n_layers` into `save_job_result({...})`. Write the file, then run it with `subprocess.run(...)` and `env={**os.environ, "N_LAYERS": "3"}` exactly as Section 4 did, and parse the script's output back into `layered_result` (printing a JSON line from the script is the tidiest way to read it back).

</details>

In [ ]:
# Exercise 3: Add an N_LAYERS hyperparameter to the entry point, then re-run the
# local subprocess check (no AWS).
# Define: layered_result -- the parsed result of running your edited entry point
#         with >= 2 layers; a dict with the counts and an n_layers field.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert isinstance(layered_result, dict), "collect the run's output into a dict"
    assert "n_layers" in layered_result, "save the layer count alongside the counts"
    assert isinstance(layered_result["n_layers"], int) and layered_result["n_layers"] >= 2, (
        "read a layer count of at least 2 from the environment"
    )
    _counts = layered_result["counts"]
    assert sum(_counts.values()) > 0, (
        "the entry point should still run its circuit and return counts"
    )
    assert all(len(b) == 2 for b in _counts), (
        "stacking H+CNOT layers keeps the circuit on two qubits"
    )

### Exercise 4 — Time the lifecycle (submitting needs AWS; the timer runs offline)

When you flip `RUN_ON_AWS`, the polling loop in Section 5 walks `QUEUED -> RUNNING -> COMPLETED`. To answer "how long did each phase take?" you must turn a stream of `(timestamp, state)` observations into a duration per phase. Factor that timing logic into a pure function so it runs and is testable with no credentials -- then it drops straight into the real polling loop.

Define `phase_durations`: a function taking a chronological list of `(timestamp, state)` tuples (as `time.time()` would produce while polling) and returning a dict mapping each phase name to the seconds spent in it.

<details><summary>Hint 1 — nudge</summary>

You do not know how long a phase lasts until you see the *next* state appear. So a phase's duration is bounded by two consecutive observations -- attribute the gap between poll i and poll i+1 to whatever state poll i reported.

</details>
<details><summary>Hint 2 — approach</summary>

Walk consecutive pairs with `zip(events, events[1:])`. For each pair `((t_a, state_a), (t_b, state_b))`, add `t_b - t_a` to a running total keyed by `state_a`. The terminal state has no successor, so it contributes nothing. To use it live, append `(time.time(), job.state())` on every poll of the `RUN_ON_AWS` loop and pass the list to your function.

</details>

In [ ]:
# Exercise 4: Turn a stream of (timestamp, state) polls into a duration per phase.
# Define: phase_durations(events) -- maps each lifecycle phase name to the seconds
#         spent in it, computed from consecutive observations.

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert callable(phase_durations), "phase_durations should be a function of the polling log"
    _log = [
        (0.0, "QUEUED"),
        (15.0, "QUEUED"),
        (45.0, "RUNNING"),
        (60.0, "RUNNING"),
        (75.0, "COMPLETED"),
    ]
    _d = phase_durations(_log)
    assert isinstance(_d, dict), "return a mapping from phase name to seconds"
    assert set(_d) >= {"QUEUED", "RUNNING"}, (
        "the lifecycle spends time in QUEUED then RUNNING before it completes"
    )
    assert all(v >= 0 for v in _d.values()), "a phase duration is never negative"
    assert abs(_d["QUEUED"] - 45.0) < 1e-6, (
        "QUEUED runs from the first QUEUED poll until the first RUNNING poll"
    )
    assert abs(_d["RUNNING"] - 30.0) < 1e-6, (
        "RUNNING runs from the first RUNNING poll until the terminal state"
    )
    assert abs(sum(_d.values()) - (_log[-1][0] - _log[0][0])) < 1e-6, (
        "the phases should tile the whole observed span"
    )

## Summary

- A Bell state (`H` then `CNOT`) produces only the correlated outcomes `|00>` and `|11>`; the parameterized `Rx(theta)+CNOT` version is the inner loop a hybrid job iterates.
- Everything that produced output here ran on the `LocalSimulator` with no AWS credentials -- the circuit, the `FreeParameter` sweep, the variational descent, and the entry point (exercised as a subprocess).
- A Hybrid Job is just your entry-point script (which calls `save_job_result`) handed to `AwsQuantumJob.create(...)`; you then poll `job.state()` through create -> QUEUED -> RUNNING -> COMPLETED and pull `job.result()` from S3. All AWS calls stay behind `RUN_ON_AWS` with a cost note.
- A job earns its keep on **iteration**: many quantum-classical steps with real queue wait, where priority access lets iterations run back-to-back. A single circuit belongs in a standalone task.

**Next:** [`02-parametric-compilation.ipynb`](02-parametric-compilation.ipynb) -- compile the parameterized circuit once and reuse it across parameter updates to cut per-iteration time.